# Lecture 10: Graph Mining
**BITE 485 | Tharaka University**

---

## Learning Outcomes
1. Build and visualise graph structures with NetworkX
2. Compute degree, betweenness, and PageRank centrality
3. Detect communities using the Louvain algorithm
4. Apply graph analysis to a transaction network fraud scenario

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
print(f'NetworkX version: {nx.__version__}')
print('Ready.')

## 10.1 Building a Graph

In [ ]:
# Create a directed weighted transaction network
G = nx.DiGraph()

# Add accounts as nodes
accounts = ['ACC_001','ACC_002','ACC_003','ACC_004','ACC_005',
            'ACC_006','ACC_007','ACC_008','ACC_009','ACC_010']
G.add_nodes_from(accounts)

# Add transactions as directed weighted edges
transactions = [
    ('ACC_001','ACC_002', 5000), ('ACC_002','ACC_003', 4800),
    ('ACC_003','ACC_001', 4600), ('ACC_001','ACC_004', 2000),  # ring: 001->002->003->001
    ('ACC_005','ACC_006',15000), ('ACC_006','ACC_007',14500),
    ('ACC_007','ACC_005',14000), ('ACC_005','ACC_008', 1000),  # ring: 005->006->007->005
    ('ACC_009','ACC_001',  500), ('ACC_009','ACC_005',  800),
    ('ACC_010','ACC_002',  300), ('ACC_010','ACC_006',  400),
    ('ACC_004','ACC_009',  200), ('ACC_008','ACC_010',  150),
]
for src, tgt, amt in transactions:
    G.add_edge(src, tgt, weight=amt)

print(f'Nodes: {G.number_of_nodes()}')
print(f'Edges: {G.number_of_edges()}')
print(f'\nIn-degree (top 3):')
for node, deg in sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:3]:
    print(f'  {node}: {deg}')

In [ ]:
# Visualise the network
plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42, k=2)
weights = [G[u][v]['weight']/3000 for u,v in G.edges()]
nx.draw_networkx_nodes(G, pos, node_size=800, node_color='steelblue', alpha=0.8)
nx.draw_networkx_labels(G, pos, font_size=8, font_color='white')
nx.draw_networkx_edges(G, pos, width=weights, arrows=True,
                        edge_color='gray', arrowsize=20, alpha=0.7)
plabels = {(u,v): f'{d["weight"]:,}' for u,v,d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=plabels, font_size=7)
plt.title('Mobile Money Transaction Network')
plt.axis('off')
plt.tight_layout()
plt.show()

## 10.2 Centrality Measures

In [ ]:
# Degree centrality
dc = nx.degree_centrality(G)
print('Degree Centrality (top 5):')
for n, s in sorted(dc.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'  {n}: {s:.4f}')

In [ ]:
# Betweenness centrality
bc = nx.betweenness_centrality(G)
print('Betweenness Centrality (top 5):')
for n, s in sorted(bc.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'  {n}: {s:.4f}')
print('\nHigh betweenness = bridge node connecting different parts of network')

In [ ]:
# PageRank
pr = nx.pagerank(G, alpha=0.85, weight='weight')
print('PageRank Scores (all nodes, sorted):')
for n, s in sorted(pr.items(), key=lambda x: x[1], reverse=True):
    print(f'  {n}: {s:.6f}')
print('\nHigh PageRank = receives value from other important nodes')

In [ ]:
# Centrality summary table
cent_df = pd.DataFrame({
    'Degree': dc, 'Betweenness': bc, 'PageRank': pr
}).round(4)
cent_df['in_degree'] = dict(G.in_degree())
cent_df['out_degree'] = dict(G.out_degree())
cent_df.sort_values('PageRank', ascending=False)

## 10.3 Cycle Detection — Finding Suspicious Rings

In [ ]:
# Find all simple cycles (closed loops of transactions)
cycles = list(nx.simple_cycles(G))
print(f'Number of cycles detected: {len(cycles)}')
for c in cycles:
    total_value = sum(G[c[i]][c[(i+1)%len(c)]]['weight'] for i in range(len(c)))
    print(f'  Cycle: {" -> ".join(c)} -> {c[0]} | Total Value: KES {total_value:,}')

## 10.4 Community Detection

In [ ]:
# Convert to undirected for community detection
G_un = G.to_undirected()

try:
    from networkx.algorithms import community
    communities = community.louvain_communities(G_un, seed=42)
    print('Communities detected by Louvain algorithm:')
    for i, comm in enumerate(communities):
        print(f'  Community {i+1}: {sorted(comm)}')
except (ImportError, AttributeError):
    # Fallback: Girvan-Newman
    from networkx.algorithms.community import girvan_newman
    comp = girvan_newman(G_un)
    communities = next(comp)
    print('Communities detected by Girvan-Newman:')
    for i, comm in enumerate(communities):
        print(f'  Community {i+1}: {sorted(comm)}')

## 10.5 Fraud Indicators Summary

In [ ]:
print('=== FRAUD ANALYSIS SUMMARY ===')
print(f'\n1. RING STRUCTURES: {len(cycles)} circular transaction pattern(s) detected')
for c in cycles:
    total = sum(G[c[i]][c[(i+1)%len(c)]]['weight'] for i in range(len(c)))
    print(f'   {" -> ".join(c)} -> {c[0]}: KES {total:,}')

high_bc = [(n,s) for n,s in bc.items() if s > 0.1]
print(f'\n2. BRIDGE NODES (high betweenness): {[n for n,s in high_bc]}')
print('   These accounts connect otherwise separate transaction clusters.')

high_pr = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:3]
print(f'\n3. HIGH PAGERANK ACCOUNTS: {[n for n,s in high_pr]}')
print('   These accounts accumulate the most influence from incoming transactions.')

### Exercise

Using the transaction network G:

1. Add 3 new accounts (ACC_011, ACC_012, ACC_013) and create a new ring transaction pattern between them with values around KES 8,000 each.
2. Re-run the cycle detection. Does the new ring appear?
3. Re-compute betweenness centrality. Which accounts have increased betweenness after adding the new transactions?
4. Create a visualisation that colour-codes nodes by their community membership.

In [ ]:
# Your code here


---
*BITE 485 Data Science | Tharaka University | kevin.tuei@tharaka.ac.ke*